In [1]:
import pandas as pd
import numpy as np

import duckdb

import pandas as pd
import numpy as np
import duckdb

db_path = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\warehouse\realestate_warehouse.duckdb"

con = duckdb.connect(db_path)

In [4]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [5]:
con.execute("""Show Tables""").df()

,name
0,anchor_zip_key_counts
1,civil_case_defendant_addr
2,civil_case_defendant_matchprep
3,civil_case_defendant_primary
4,civil_case_defendants
5,civil_case_defendants_clean
6,civil_case_defendants_dedup
7,civil_case_features
8,civil_case_parties_clean
9,civil_case_property_clean


In [10]:
dfcivilcase = con.execute("""Select*
FROM civil_cases_raw""").df()

dfcivilevent = con.execute("""SELECT*
FROM civil_events_raw""").df()

dfcivilparty = con.execute("""SELECT*
FROM civil_parties_raw""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
dfcivilcase.head(3)

,CaseNumber,FileDate,Nature_of_Proceeding,CourtNo,Status
0,1000008,2019-07-19 00:00:00,Contract - Consumer/Commercial/Debt,2,Closed
1,1000164,2017-04-18 00:00:00,Contract - Consumer/Commercial/Debt,2,Closed
2,1000329,2020-08-24 00:00:00,Contract - Consumer/Commercial/Debt,3,Closed


In [12]:
dfcivilevent.head(3)

,CaseNumber,DocumentID,FileDate,EventDesc,Comments,Number of Pages
0,472896-401,2025.12.04 Application for Removal of Guardian...,2025-12-11 00:00:00,eFile Original Petition,Application to Remove Guardian and for Appoint...,69
1,000000,Oath,2017-11-07 00:00:00,Oath,COURT INVESTIGATOR OATH,1
2,000752,Report of Sale,2023-11-30 00:00:00,Reports of Miscellaneous Kinds,None,12


In [13]:
dfcivilparty.head(3)

,CaseNumber,PartyRole,PName,PAddr,PAddr2
0,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
1,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
2,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None


In [15]:
dfcivilcase[dfcivilcase['CaseNumber'] == '1000008']

,CaseNumber,FileDate,Nature_of_Proceeding,CourtNo,Status
0,1000008,2019-07-19 00:00:00,Contract - Consumer/Commercial/Debt,2,Closed


In [16]:
dfcivilevent[dfcivilevent['CaseNumber'] == '1000008']

,CaseNumber,DocumentID,FileDate,EventDesc,Comments,Number of Pages
618,1000008,132000.pdf,2019-07-19 00:00:00,Answer,None,13
619,1000008,Motion to Withdraw Counsel,2019-07-19 00:00:00,Motion to Withdraw Counsel,PLAINTIFF'S MOTION TO WITHDRAW COUNSEL AND SUB...,None
621,1000008,Civil Case Information Sheet,2019-07-19 00:00:00,Civil Case Information Sheet,None,1
622,1000008,14-95308 nbg.pdf,2019-07-19 00:00:00,Notice,of Bank Garnishment,11
623,1000008,Letter,2019-07-19 00:00:00,Letter,FROM FULTON FRIEDMAN & GULLACE,None
625,1000008,Order for Substituted Service,2019-07-19 00:00:00,Order for Substituted Service,GRANTED,None
627,1000008,Writ of Garnishment After Judgment,2019-07-19 00:00:00,Writ of Garnishment After Judgment Issued,m/o to atty,3
628,1000008,IRENE C GUEVARA 14-95308.pdf,2019-07-19 00:00:00,Application for Garnishment After Judgment (OCA),None,5
629,1000008,Order to Substitute Counsel,2019-07-19 00:00:00,Order to Substitute Counsel,ORDER GRANTING WITHDRAWAL OF COUNSEL AND SUBST...,None
631,1000008,Motion to Withdraw Counsel,2019-07-19 00:00:00,Motion to Withdraw Counsel,OF RECORD FOR SUBSTITUTION OF COUNSEL AND DESI...,None


In [17]:
dfcivilparty[dfcivilparty['CaseNumber'] == '1000008']

,CaseNumber,PartyRole,PName,PAddr,PAddr2
0,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
1,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
2,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
3,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
4,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
5,1000008,Plaintiff,ASSET ACCEPTANCE LLC,None,None
6,1000008,Defendant,"GUEVARA, IRENE C",5114 MAURITZ DR,"HOUSTON, TX 77032"
7,1000008,Garnishee,"WELLS FARGO BANK, N.A","C/O CORPORATION SERVICE CO. 701 BRAZOS, SUITE ...","AUSTIN, TX 78701"
8,1000008,Garnishor,ASSET ACCEPTANCE LLC,None,None
9,1000008,Garnishor,ASSET ACCEPTANCE LLC,None,None


In [18]:
dfdefend = con.execute("""select* from civil_case_defendants""").df()

dfdefend.head(10)

,CaseNumber,defendant_name,defendant_addr1,defendant_addr2
0,1258421,"U.S. Gulf Coast Trading Co., Inc.",14429 Auto Park Way,"Houston, TX 77083"
1,1258478,"Martland, Dytonya",4817 BRINKLEY ST,"HOUSTON, TX 77033-3805"
2,1258485,"Gordon, Richard, Jr.",12018 Legend Manor,"Houston, TX 77082"
3,1258486,All Other Occupants,706 Lexus Dr.,"Humble, TX 77396"
4,1258498,Madina Construction LLC,c/o Javeed Hyder 2103 Nob Hill,"Carrollton, TX 75006"
5,1258535,"Sims, Naomi Nicole","14550 Fonmeadow Dr, Unit 1202","Houston, TX 77035"
6,1258569,All Other Occupants,"3400 Shady Hill Dr., Apt 08A","Baytown, TX 77521"
7,1258605,All Occupants,None,None
8,1258612,"Deal, Dustin J.",6101 Southwest Fwy Ste 213,"Houston, TX 77057"
9,1258615,et al,13480 S. Thorntree #313,"Houston, TX 77015"


In [19]:
con.execute("""
CREATE OR REPLACE TABLE civil_case_summary AS
WITH event_flags AS (
    SELECT
        CaseNumber,

        COUNT(*) AS total_civil_events,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%default judgment%' 
              OR lower(DocumentID) LIKE '%default judgment%'
            THEN 1 ELSE 0 
        END) AS has_default_judgment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%garnishment%' 
              OR lower(DocumentID) LIKE '%garnishment%'
              OR lower(Comments) LIKE '%garnishment%'
            THEN 1 ELSE 0 
        END) AS has_garnishment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%final judgment%' 
              OR lower(DocumentID) LIKE '%final judgment%'
            THEN 1 ELSE 0 
        END) AS has_final_judgment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%judgment%' 
              OR lower(DocumentID) LIKE '%judgment%'
            THEN 1 ELSE 0 
        END) AS has_any_judgment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%eviction%' 
              OR lower(DocumentID) LIKE '%eviction%'
            THEN 1 ELSE 0 
        END) AS has_eviction,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%foreclosure%' 
              OR lower(DocumentID) LIKE '%foreclosure%'
            THEN 1 ELSE 0 
        END) AS has_foreclosure

    FROM dfcivilevent
    GROUP BY CaseNumber
),

party_summary AS (
    SELECT
        CaseNumber,

        COUNT(*) AS total_parties,

        MAX(CASE WHEN lower(PartyRole) = 'defendant' THEN 1 ELSE 0 END) AS has_defendant,

        STRING_AGG(DISTINCT CASE 
            WHEN lower(PartyRole) = 'defendant' THEN PName 
        END, '; ') AS defendant_names,

        STRING_AGG(DISTINCT CASE 
            WHEN lower(PartyRole) = 'defendant' THEN PAddr 
        END, '; ') AS defendant_addresses,

        STRING_AGG(DISTINCT CASE 
            WHEN lower(PartyRole) = 'defendant' THEN PAddr2 
        END, '; ') AS defendant_city_state_zip

    FROM dfcivilparty
    GROUP BY CaseNumber
)

SELECT
    c.CaseNumber,
    CAST(c.FileDate AS DATE) AS case_file_date,
    c.Nature_of_Proceeding,
    c.CourtNo,
    c.Status,

    COALESCE(e.total_civil_events, 0) AS total_civil_events,

    COALESCE(e.has_default_judgment, 0) AS has_default_judgment,
    COALESCE(e.has_garnishment, 0) AS has_garnishment,
    COALESCE(e.has_final_judgment, 0) AS has_final_judgment,
    COALESCE(e.has_any_judgment, 0) AS has_any_judgment,
    COALESCE(e.has_eviction, 0) AS has_eviction,
    COALESCE(e.has_foreclosure, 0) AS has_foreclosure,

    (
        3 * COALESCE(e.has_garnishment, 0)
      + 3 * COALESCE(e.has_default_judgment, 0)
      + 2 * COALESCE(e.has_final_judgment, 0)
      + 1 * COALESCE(e.has_any_judgment, 0)
      + 3 * COALESCE(e.has_eviction, 0)
      + 3 * COALESCE(e.has_foreclosure, 0)
    ) AS civil_severity_score,

    COALESCE(p.total_parties, 0) AS total_parties,
    COALESCE(p.has_defendant, 0) AS has_defendant,
    p.defendant_names,
    p.defendant_addresses,
    p.defendant_city_state_zip

FROM dfcivilcase c
LEFT JOIN event_flags e
    ON c.CaseNumber = e.CaseNumber
LEFT JOIN party_summary p
    ON c.CaseNumber = p.CaseNumber
""")

In [22]:
con.execute("""
CREATE OR REPLACE TABLE civil_case_summary AS
WITH event_flags AS (
    SELECT
        CaseNumber,

        COUNT(*) AS total_case_document_filings,

        COUNT(*) FILTER (
            WHERE lower(EventDesc) LIKE '%judgment%'
               OR lower(EventDesc) LIKE '%garnishment%'
        ) AS num_high_severity_filings,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%default judgment%' 
              OR lower(DocumentID) LIKE '%default judgment%'
            THEN 1 ELSE 0 
        END) AS has_default_judgment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%garnishment%' 
              OR lower(DocumentID) LIKE '%garnishment%'
              OR lower(Comments) LIKE '%garnishment%'
            THEN 1 ELSE 0 
        END) AS has_garnishment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%final judgment%' 
              OR lower(DocumentID) LIKE '%final judgment%'
            THEN 1 ELSE 0 
        END) AS has_final_judgment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%judgment%' 
              OR lower(DocumentID) LIKE '%judgment%'
            THEN 1 ELSE 0 
        END) AS has_any_judgment,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%eviction%' 
              OR lower(DocumentID) LIKE '%eviction%'
            THEN 1 ELSE 0 
        END) AS has_eviction,

        MAX(CASE 
            WHEN lower(EventDesc) LIKE '%foreclosure%' 
              OR lower(DocumentID) LIKE '%foreclosure%'
            THEN 1 ELSE 0 
        END) AS has_foreclosure

    FROM dfcivilevent
    GROUP BY CaseNumber
),

party_summary AS (
    SELECT
        CaseNumber,

        COUNT(*) AS total_parties,

        MAX(CASE WHEN lower(PartyRole) = 'defendant' THEN 1 ELSE 0 END) AS has_defendant,

        STRING_AGG(DISTINCT CASE 
            WHEN lower(PartyRole) = 'defendant' THEN PName 
        END, '; ') AS defendant_names,

        STRING_AGG(DISTINCT CASE 
            WHEN lower(PartyRole) = 'defendant' THEN PAddr 
        END, '; ') AS defendant_addresses,

        STRING_AGG(DISTINCT CASE 
            WHEN lower(PartyRole) = 'defendant' THEN PAddr2 
        END, '; ') AS defendant_city_state_zip

    FROM dfcivilparty
    GROUP BY CaseNumber
)

SELECT
    c.CaseNumber,
    CAST(c.FileDate AS DATE) AS case_file_date,
    c.Nature_of_Proceeding,
    c.CourtNo,
    c.Status,

    COALESCE(e.total_case_document_filings, 0) AS total_case_document_filings,
    COALESCE(e.num_high_severity_filings, 0) AS num_high_severity_filings,

    COALESCE(e.has_default_judgment, 0) AS has_default_judgment,
    COALESCE(e.has_garnishment, 0) AS has_garnishment,
    COALESCE(e.has_final_judgment, 0) AS has_final_judgment,
    COALESCE(e.has_any_judgment, 0) AS has_any_judgment,
    COALESCE(e.has_eviction, 0) AS has_eviction,
    COALESCE(e.has_foreclosure, 0) AS has_foreclosure,

    (
        3 * COALESCE(e.has_garnishment, 0)
      + 3 * COALESCE(e.has_default_judgment, 0)
      + 2 * COALESCE(e.has_final_judgment, 0)
      + 1 * COALESCE(e.has_any_judgment, 0)
      + 3 * COALESCE(e.has_eviction, 0)
      + 3 * COALESCE(e.has_foreclosure, 0)
    ) AS civil_severity_score,

    COALESCE(p.total_parties, 0) AS total_parties,
    COALESCE(p.has_defendant, 0) AS has_defendant,
    p.defendant_names,
    p.defendant_addresses,
    p.defendant_city_state_zip

FROM dfcivilcase c
LEFT JOIN event_flags e
    ON c.CaseNumber = e.CaseNumber
LEFT JOIN party_summary p
    ON c.CaseNumber = p.CaseNumber
""")

In [23]:
con.execute("""
SELECT *
FROM civil_case_summary
WHERE CaseNumber = '1000008'
""").df()

,CaseNumber,case_file_date,Nature_of_Proceeding,CourtNo,Status,total_case_document_filings,num_high_severity_filings,has_default_judgment,has_garnishment,has_final_judgment,has_any_judgment,has_eviction,has_foreclosure,civil_severity_score,total_parties,has_defendant,defendant_names,defendant_addresses,defendant_city_state_zip
0,1000008,2019-07-19,Contract - Consumer/Commercial/Debt,2,Closed,23,9,1,1,1,1,0,0,9,10,1,"GUEVARA, IRENE C",5114 MAURITZ DR,"HOUSTON, TX 77032"


In [24]:
con.execute("""
SELECT 
    civil_severity_score,
    COUNT(*) AS case_count
FROM civil_case_summary
GROUP BY civil_severity_score
ORDER BY civil_severity_score DESC
""").df()

,civil_severity_score,case_count
0,12,9
1,10,1
2,9,1663
3,7,3163
4,6,22039
5,4,77642
6,3,9872
7,1,23352
8,0,83357


In [25]:
gold_path = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\realestate_gold.duckdb"

con.execute(f"""
ATTACH '{gold_path}' AS gold
""")

In [26]:
con.execute("SHOW DATABASES").df()

,database_name
0,gold
1,realestate_warehouse


In [27]:
con.execute("""
SHOW TABLES FROM gold
""").df()

,name
0,appraisals
1,civil_case_defendant_primary
2,civil_case_features
3,civil_case_owner_match
4,civil_case_property_clean
5,current_owner
6,deed_financial_events
7,deed_financial_events_raw
8,deed_financial_features
9,deed_financial_features_enhanced


In [29]:
dfanch = con.execute("""Select* from property_anchor_with_legal""").df()

dfanch.head(15)

,acct,site_addr_1,site_addr_2,site_addr_3,str_pfx,str_num,str_num_sfx,str,str_sfx,str_sfx_dir,str_unit,mailto,mail_addr_1,mail_addr_2,mail_city,mail_state,mail_zip,mail_country,undeliverable,situs_full,mail_full,addr_key,addr_key_no_unit,owner_entity_flag,absentee_owner_flag,situs_zip,addr_zip_key,state_class,school_dist,map_facet,key_map,Neighborhood_Code,Neighborhood_Grp,Market_Area_1,Market_Area_1_Dscr,Market_Area_2,Market_Area_2_Dscr,econ_area,econ_bld_class,center_code,yr_impr,yr_annexed,splt_dt,dsc_cd,nxt_bld,bld_ar,land_ar,acreage,Cap_acct,shared_cad,lgl_1,lgl_2,lgl_3,lgl_4,jurs,legal_full_raw,legal_full_clean
0,0010010000013,0 COMMERCE ST,HOUSTON,77002,None,0,None,COMMERCE,ST,None,None,CITY OF HOUSTON,PO BOX 1562,None,HOUSTON,TX,77251-1562,None,N,0 COMMERCE ST HOUSTON 77002,PO BOX 1562 HOUSTON TX 77251-1562,0 COMMERCE ST,0 COMMERCE ST,False,True,77002,0 COMMERCE ST|77002,X1,1,5457A,493L,5900.05,0,4001,Central Business District,4001,Central Business District,1,E,28,<NA>,None,None,None,1,0,44431.0,1.0200,N,N,ALL BLK 1,SSBB,None,None,001 040 041 042 043 044 048 061 265 268 576,ALL BLK 1 SSBB,ALL BLK 1 SSBB
1,0010020000001,907 COMMERCE ST,HOUSTON,77002,None,907,None,COMMERCE,ST,None,None,CURRENT OWNER,1717 SAINT JAMES PLACE STE 112,None,HOUSTON,TX,77056-3412,None,N,907 COMMERCE ST HOUSTON 77002,1717 SAINT JAMES PLACE STE 112 HOUSTON TX 770...,907 COMMERCE ST,907 COMMERCE ST,False,True,77002,907 COMMERCE ST|77002,F1,1,5457A,493M,5900.05,0,5001,Central Business District,4001,Central Business District,1,E,28,<NA>,None,None,None,1,0,5001.0,0.1148,N,N,TR 15 BLK 2,SSBB,None,None,001 040 041 042 043 044 048 061 265 268 576,TR 15 BLK 2 SSBB,TR 15 BLK 2 SSBB
2,0010020000003,0 COMMERCE ST,HOUSTON,77002,None,0,None,COMMERCE,ST,None,None,MILBY CHARLES FAMILY PTNSH,2612 TODVILLE RD,None,SEABROOK,TX,77586-3008,None,N,0 COMMERCE ST HOUSTON 77002,2612 TODVILLE RD SEABROOK TX 77586-3008,0 COMMERCE ST,0 COMMERCE ST,False,True,77002,0 COMMERCE ST|77002,F1,1,5457A,493M,5900.05,0,4001,Central Business District,4001,Central Business District,1,E,33,<NA>,None,None,None,1,0,18121.0,0.4160,N,N,.500 U/D INT IN TR 14 BLK 2,SSBB,None,None,001 040 041 042 043 044 048 061 265 268 576,.500 U/D INT IN TR 14 BLK 2 SSBB,500 U D INT IN TR 14 BLK 2 SSBB
3,0010020000004,0 COMMERCE ST,HOUSTON,77002,None,0,None,COMMERCE,ST,None,None,MICHAEL RYAN FEAGIN TRUST,3302 SUFFOLK DR,None,HOUSTON,TX,77027-6326,None,N,0 COMMERCE ST HOUSTON 77002,3302 SUFFOLK DR HOUSTON TX 77027-6326,0 COMMERCE ST,0 COMMERCE ST,True,True,77002,0 COMMERCE ST|77002,F1,1,5457A,493M,5900.05,0,4001,Central Business District,4001,Central Business District,1,E,33,<NA>,None,None,None,1,0,9061.0,0.2080,N,N,.25 U/D INT IN TR 14 BLK 2,SSBB,None,None,001 040 041 042 043 044 048 061 265 268 576,.25 U/D INT IN TR 14 BLK 2 SSBB,25 U D INT IN TR 14 BLK 2 SSBB
4,0010020000013,921 COMMERCE ST,HOUSTON,77002,None,921,None,COMMERCE,ST,None,None,BUFFALO BAYOU PARTNERSHIP,1019 COMMERCE ST STE 200,None,HOUSTON,TX,77002-1701,None,N,921 COMMERCE ST HOUSTON 77002,1019 COMMERCE ST STE 200 HOUSTON TX 77002-1701,921 COMMERCE ST,921 COMMERCE ST,True,True,77002,921 COMMERCE ST|77002,X1,1,5457A,493M,5900.05,0,4001,Central Business District,4001,Central Business District,1,E,28,<NA>,None,None,None,1,0,3001.0,0.0689,N,N,TR 13 BLK 2,SSBB,None,None,001 040 041 042 043 044 048 061 265 268 576,TR 13 BLK 2 SSBB,TR 13 BLK 2 SSBB
5,0010020000015,909 COMMERCE ST,HOUSTON,77002,None,909,None,COMMERCE,ST,None,None,BUFFALO BAYOU PARTNERSHIP ATTN: ANNE OLSON,1019 COMMERCE ST STE 200,None,HOUSTON,TX,77002-1701,None,N,909 COMMERCE ST HOUSTON 77002,1019 COMMERCE ST STE 200 HOUSTON TX 77002-1701,909 COMMERCE ST,909 COMMERCE ST,True,True,77002,909 COMMERCE ST|77002,X1,1,5457A,493M,5900.05,0,4001,Central Business District,4001,Central Business District,1,E,28,<NA>,None,None,None,1,0,1250.0,0.0287,N,N,TR 15A BLK 2,SSBB,None,None,001 040 041 042 043 044 048 061 265 268 576,TR 15A BLK 2 SSBB,TR 15A BLK 2 SSBB
6,0010020000016,901 COMMERCE ST,HOUS

In [31]:
con.execute("""
CREATE OR REPLACE TABLE gold.civil_case_summary AS
SELECT *
FROM civil_case_summary
""")

In [32]:
con.execute("""
CREATE OR REPLACE TABLE gold.civil_case_summary_with_keys AS
SELECT
    *,

    -- clean street
    upper(trim(defendant_addresses)) AS civil_addr_clean,

    -- extract zip
    regexp_extract(defendant_city_state_zip, '(\\d{5})', 1) AS civil_zip,

    -- build join key
    upper(trim(defendant_addresses)) || '|' ||
    regexp_extract(defendant_city_state_zip, '(\\d{5})', 1) AS civil_addr_zip_key

FROM gold.civil_case_summary
""")

In [33]:
con.execute("""
CREATE OR REPLACE TABLE gold.civil_case_property_match AS
SELECT
    c.*,
    a.acct,

    CASE 
        WHEN a.acct IS NOT NULL THEN 1
        ELSE 0
    END AS matched_flag

FROM gold.civil_case_summary_with_keys c
LEFT JOIN gold.property_anchor_with_legal a
    ON c.civil_addr_zip_key = a.addr_zip_key
""")

In [ ]:
#match rate of civil court cases was 20%
con.execute("""
SELECT 
    matched_flag,
    COUNT(*) AS cnt
FROM gold.civil_case_property_match
GROUP BY matched_flag
""").df()

,matched_flag,cnt
0,0,179929
1,1,45471


In [35]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CaseNumber) AS distinct_cases,
    COUNT(DISTINCT acct) AS distinct_properties
FROM gold.civil_case_property_match
WHERE matched_flag = 1
""").df()

,total_rows,distinct_cases,distinct_properties
0,45471,41169,38625


In [ ]:
#massive duplicate events...
con.execute("""
SELECT
    CaseNumber,
    COUNT(DISTINCT acct) AS matched_acct_count
FROM gold.civil_case_property_match
WHERE matched_flag = 1
GROUP BY CaseNumber
HAVING COUNT(DISTINCT acct) > 1
ORDER BY matched_acct_count DESC
LIMIT 25
""").df()

,CaseNumber,matched_acct_count
0,1156509,296
1,1137452,294
2,1138974,251
3,1158690,248
4,1141548,241
5,1126642,235
6,1062839,180
7,1096523,179
8,1114122,150
9,1086120,149


In [37]:
con.execute("""
SELECT
    CaseNumber,
    defendant_addresses,
    defendant_city_state_zip,
    civil_addr_zip_key,
    acct
FROM gold.civil_case_property_match
WHERE CaseNumber = '1156509'
ORDER BY acct
LIMIT 50
""").df()

,CaseNumber,defendant_addresses,defendant_city_state_zip,civil_addr_zip_key,acct
0,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010001
1,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010002
2,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010003
3,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010004
4,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010005
5,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010006
6,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010007
7,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800010008
8,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800020001
9,1156509,9700 LEAWOOD BLVD,"Houston, TX 77099-2531",9700 LEAWOOD BLVD|77099,1155800020002


In [38]:
con.execute("""
CREATE OR REPLACE TABLE gold.civil_case_property_match_deduped AS
WITH match_counts AS (
    SELECT
        CaseNumber,
        COUNT(DISTINCT acct) AS matched_acct_count
    FROM gold.civil_case_property_match
    WHERE matched_flag = 1
    GROUP BY CaseNumber
),

clean_matches AS (
    SELECT
        m.*
    FROM gold.civil_case_property_match m
    JOIN match_counts mc
        ON m.CaseNumber = mc.CaseNumber
    WHERE m.matched_flag = 1
      AND mc.matched_acct_count = 1
)

SELECT *
FROM clean_matches
""")

In [ ]:
#one row per case after deduping
con.execute("""
SELECT
    COUNT(*) AS rows_after_deduping,
    COUNT(DISTINCT CaseNumber) AS distinct_cases,
    COUNT(DISTINCT acct) AS distinct_properties
FROM gold.civil_case_property_match_deduped
""").df()

,rows_after_deduping,distinct_cases,distinct_properties
0,40633,40633,33984


In [40]:
con.execute("""
CREATE OR REPLACE TABLE gold.ownership_window_civil_cases AS
SELECT
    oh.*,

    c.CaseNumber,
    c.case_file_date,
    c.Nature_of_Proceeding,
    c.Status,
    c.total_case_document_filings,
    c.num_high_severity_filings,
    c.has_default_judgment,
    c.has_garnishment,
    c.has_final_judgment,
    c.has_any_judgment,
    c.has_eviction,
    c.has_foreclosure,
    c.civil_severity_score,
    c.defendant_names,
    c.defendant_addresses,
    c.defendant_city_state_zip,
    c.civil_addr_zip_key

FROM gold.ownership_history oh
JOIN gold.civil_case_property_match_deduped c
    ON oh.acct = c.acct
   AND c.case_file_date >= oh.start_date
   AND c.case_file_date < COALESCE(oh.end_date, DATE '2999-12-31')
""")

In [41]:
#ownership window for civil cases
con.execute("""
CREATE OR REPLACE TABLE gold.ownership_window_civil_events AS
SELECT
    oh.acct,
    oh.start_date,
    oh.end_date,

    c.CaseNumber,
    c.case_file_date,
    c.civil_severity_score,
    c.total_case_document_filings,
    c.num_high_severity_filings,
    c.has_garnishment,
    c.has_default_judgment,
    c.has_final_judgment

FROM gold.ownership_history oh
JOIN gold.civil_case_property_match_deduped c
    ON oh.acct = c.acct
   AND c.case_file_date >= oh.start_date
   AND c.case_file_date < COALESCE(oh.end_date, DATE '2999-12-31')
""")

In [42]:
#aggregate ownership into windows
con.execute("""
CREATE OR REPLACE TABLE gold.ownership_window_civil_features AS
SELECT
    acct,
    start_date,
    end_date,

    COUNT(*) AS num_civil_cases,

    SUM(civil_severity_score) AS total_civil_severity,
    MAX(civil_severity_score) AS max_civil_severity,

    SUM(has_garnishment) AS num_garnishments,
    SUM(has_default_judgment) AS num_default_judgments,
    SUM(has_final_judgment) AS num_final_judgments,

    AVG(total_case_document_filings) AS avg_case_filings,

    MAX(case_file_date) AS last_civil_case_date

FROM gold.ownership_window_civil_events
GROUP BY acct, start_date, end_date
""")

In [44]:
dfcivilgold = con.execute("""Select* from gold.ownership_window_civil_features""").df()

dfcivilgold.head(10)

,acct,start_date,end_date,num_civil_cases,total_civil_severity,max_civil_severity,num_garnishments,num_default_judgments,num_final_judgments,avg_case_filings,last_civil_case_date
0,0162550070036,2020-05-07,NaT,1,0.0,0,0.0,0.0,0.0,25.0,2025-02-18
1,0640340060021,2015-03-18,2021-03-31,1,4.0,4,0.0,1.0,0.0,17.0,2015-11-11
2,0791410050125,2016-12-19,NaT,1,4.0,4,0.0,1.0,0.0,21.0,2018-07-11
3,1144630020006,2018-04-13,NaT,1,4.0,4,0.0,1.0,0.0,26.0,2018-11-16
4,0770010000011,2015-03-10,NaT,1,1.0,1,0.0,0.0,0.0,17.0,2019-07-10
5,0540220000030,2021-04-21,NaT,1,4.0,4,0.0,1.0,0.0,24.0,2022-06-27
6,0960090000040,2019-11-19,NaT,1,4.0,4,0.0,1.0,0.0,17.0,2020-02-28
7,1181500010011,2017-07-12,2022-11-21,1,4.0,4,0.0,1.0,0.0,9.0,2021-04-05
8,0331330960005,2020-08-11,NaT,1,4.0,4,0.0,1.0,0.0,18.0,2024-10-01
9,0520570320030,2021-12-15,NaT,1,0.0,0,0.0,0.0,0.0,20.0,2024-11-06


In [45]:
dfcivilgold[dfcivilgold['acct'] == '0640340060021']

,acct,start_date,end_date,num_civil_cases,total_civil_severity,max_civil_severity,num_garnishments,num_default_judgments,num_final_judgments,avg_case_filings,last_civil_case_date
1,0640340060021,2015-03-18,2021-03-31,1,4.0,4,0.0,1.0,0.0,17.0,2015-11-11


In [46]:
#adding days since last civil filing

con.execute("""
CREATE OR REPLACE TABLE gold.ownership_window_civil_features AS
SELECT
    acct,
    start_date,
    end_date,

    COUNT(*) AS num_civil_cases,

    SUM(civil_severity_score) AS total_civil_severity,
    MAX(civil_severity_score) AS max_civil_severity,

    SUM(has_garnishment) AS num_garnishments,
    SUM(has_default_judgment) AS num_default_judgments,
    SUM(has_final_judgment) AS num_final_judgments,

    AVG(total_case_document_filings) AS avg_case_filings,

    MAX(case_file_date) AS last_civil_case_date,

    -- 🔥 NEW (CRITICAL)
    DATEDIFF('day',
        MAX(case_file_date),
        COALESCE(end_date, CURRENT_DATE)
    ) AS days_since_last_civil

FROM gold.ownership_window_civil_events
GROUP BY acct, start_date, end_date
""")

In [47]:
dfcivilgold = con.execute("""Select* from gold.ownership_window_civil_features""").df()

dfcivilgold.head(10)

,acct,start_date,end_date,num_civil_cases,total_civil_severity,max_civil_severity,num_garnishments,num_default_judgments,num_final_judgments,avg_case_filings,last_civil_case_date,days_since_last_civil
0,0162550070036,2020-05-07,NaT,1,0.0,0,0.0,0.0,0.0,25.0,2025-02-18,432
1,0640340060021,2015-03-18,2021-03-31,1,4.0,4,0.0,1.0,0.0,17.0,2015-11-11,1967
2,0791410050125,2016-12-19,NaT,1,4.0,4,0.0,1.0,0.0,21.0,2018-07-11,2846
3,1144630020006,2018-04-13,NaT,1,4.0,4,0.0,1.0,0.0,26.0,2018-11-16,2718
4,0770010000011,2015-03-10,NaT,1,1.0,1,0.0,0.0,0.0,17.0,2019-07-10,2482
5,0540220000030,2021-04-21,NaT,1,4.0,4,0.0,1.0,0.0,24.0,2022-06-27,1399
6,0960090000040,2019-11-19,NaT,1,4.0,4,0.0,1.0,0.0,17.0,2020-02-28,2249
7,1181500010011,2017-07-12,2022-11-21,1,4.0,4,0.0,1.0,0.0,9.0,2021-04-05,595
8,0331330960005,2020-08-11,NaT,1,4.0,4,0.0,1.0,0.0,18.0,2024-10-01,572
9,0520570320030,2021-12-15,NaT,1,0.0,0,0.0,0.0,0.0,20.0,2024-11-06,536


In [48]:
#adding zero case windows
con.execute("""
CREATE OR REPLACE TABLE gold.ownership_window_civil_features_full AS
SELECT
    oh.acct,
    oh.start_date,
    oh.end_date,

    COALESCE(f.num_civil_cases, 0) AS num_civil_cases,
    COALESCE(f.total_civil_severity, 0) AS total_civil_severity,
    COALESCE(f.max_civil_severity, 0) AS max_civil_severity,

    COALESCE(f.num_garnishments, 0) AS num_garnishments,
    COALESCE(f.num_default_judgments, 0) AS num_default_judgments,
    COALESCE(f.num_final_judgments, 0) AS num_final_judgments,

    COALESCE(f.avg_case_filings, 0) AS avg_case_filings,

    f.last_civil_case_date,

    -- if no case, set large value (or NULL depending on modeling choice)
    COALESCE(
        f.days_since_last_civil,
        DATEDIFF('day', oh.start_date, COALESCE(oh.end_date, CURRENT_DATE))
    ) AS days_since_last_civil

FROM gold.ownership_history oh
LEFT JOIN gold.ownership_window_civil_features f
    ON oh.acct = f.acct
   AND oh.start_date = f.start_date
   AND (oh.end_date IS NOT DISTINCT FROM f.end_date)
""")

In [49]:
dfcivilgold = con.execute("""Select* from gold.ownership_window_civil_features""").df()

dfcivilgold.head(10)

,acct,start_date,end_date,num_civil_cases,total_civil_severity,max_civil_severity,num_garnishments,num_default_judgments,num_final_judgments,avg_case_filings,last_civil_case_date,days_since_last_civil
0,0162550070036,2020-05-07,NaT,1,0.0,0,0.0,0.0,0.0,25.0,2025-02-18,432
1,0640340060021,2015-03-18,2021-03-31,1,4.0,4,0.0,1.0,0.0,17.0,2015-11-11,1967
2,0791410050125,2016-12-19,NaT,1,4.0,4,0.0,1.0,0.0,21.0,2018-07-11,2846
3,1144630020006,2018-04-13,NaT,1,4.0,4,0.0,1.0,0.0,26.0,2018-11-16,2718
4,0770010000011,2015-03-10,NaT,1,1.0,1,0.0,0.0,0.0,17.0,2019-07-10,2482
5,0540220000030,2021-04-21,NaT,1,4.0,4,0.0,1.0,0.0,24.0,2022-06-27,1399
6,0960090000040,2019-11-19,NaT,1,4.0,4,0.0,1.0,0.0,17.0,2020-02-28,2249
7,1181500010011,2017-07-12,2022-11-21,1,4.0,4,0.0,1.0,0.0,9.0,2021-04-05,595
8,0331330960005,2020-08-11,NaT,1,4.0,4,0.0,1.0,0.0,18.0,2024-10-01,572
9,0520570320030,2021-12-15,NaT,1,0.0,0,0.0,0.0,0.0,20.0,2024-11-06,536
